# ML Model Monitoring Pipeline

---

## Overview

Once a machine learning model is deployed, its performance can quietly degrade as the real-world data it encounters shifts away from the data it was trained on. This is called **data drift**, and it is one of the most common and least-visible causes of model failure in production systems.

This notebook builds an end-to-end monitoring pipeline that:

1. Trains a baseline classifier and logs it with **MLflow**
2. Simulates three monthly batches of incoming data with increasing drift
3. Generates an automated **Evidently** data drift report
4. Tracks performance metrics across batches using MLflow experiment tracking
5. Visualises accuracy and AUC-ROC degradation over time
6. Fires retrain alerts when performance drops below a defined threshold

### Dataset
The [Wisconsin Breast Cancer Dataset](https://scikit-learn.org/stable/datasets/toy_dataset.html#breast-cancer-dataset) (built into scikit-learn) is used as a clinical outcome classification proxy — 569 samples, 30 numeric features, binary target (malignant vs benign).

### Tools
| Tool | Role |
|---|---|
| MLflow | Experiment tracking, parameter/metric logging, model registry |
| Evidently | Statistical data drift detection, HTML reporting |
| Scikit-learn | Model training and evaluation |
| Pandas / NumPy | Data manipulation and drift simulation |
| Matplotlib | Performance visualisation |

## Setup

Run this cell first every time. It installs all dependencies into the active kernel, silences MLflow and Git warnings, and sets the correct matplotlib backend.

In [1]:
import subprocess, sys, os, warnings, logging

# Install all dependencies into THIS kernel's Python
subprocess.check_call([
    sys.executable, "-m", "pip", "install",
    "mlflow", "scikit-learn", "pandas", "numpy",
    "matplotlib>=3.9", "evidently",
    "--upgrade", "-q"
])

# Silence MLflow Git warnings and deprecation notices
os.environ["GIT_PYTHON_REFRESH"] = "quiet"
warnings.filterwarnings("ignore")
logging.getLogger("mlflow").setLevel(logging.ERROR)

# Set matplotlib backend BEFORE importing pyplot
# Use 'Agg' for saving files; change to 'TkAgg' if you want interactive windows
import matplotlib
matplotlib.use('Agg')

# Create output directories
os.makedirs("reports", exist_ok=True)
os.makedirs("plots",   exist_ok=True)

print("Setup complete.")

Setup complete.


---
## Section 1 — Train and Log Baseline Model

We load the breast cancer dataset, split it 80/20, and train a Random Forest classifier. All hyperparameters, metrics, and the serialised model are logged to an MLflow experiment called `clinical-outcome-monitoring`.

**Why MLflow?** It acts as an audit trail — every run is timestamped and stored with full context so you can compare, reproduce, or roll back any result.  
After running this cell, open a terminal and run `mlflow ui`, then go to `http://localhost:5000` to see the dashboard.

In [2]:
import mlflow
import mlflow.sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Load dataset
data = load_breast_cancer()
X    = pd.DataFrame(data.data, columns=data.feature_names)
y    = data.target

print(f"Dataset shape : {X.shape}")
print(f"Class balance : {pd.Series(y).value_counts().to_dict()}  (0=malignant, 1=benign)")

# Train / test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

# Train and log with MLflow
mlflow.set_experiment("clinical-outcome-monitoring")

with mlflow.start_run(run_name="baseline_model"):
    rf    = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)

    preds = rf.predict(X_test)
    acc   = accuracy_score(y_test, preds)
    auc   = roc_auc_score(y_test, rf.predict_proba(X_test)[:, 1])

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("random_state", 42)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("auc_roc",  auc)
    mlflow.sklearn.log_model(rf, "model")

    print(f"\nBaseline model logged to MLflow")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  AUC-ROC  : {auc:.4f}")

Dataset shape : (569, 30)
Class balance : {1: 357, 0: 212}  (0=malignant, 1=benign)


2026/09/09 09:45:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



Baseline model logged to MLflow
  Accuracy : 0.9649
  AUC-ROC  : 0.9953


---
## Section 2 — Simulate Data Drift

In production, feature distributions shift over time — due to seasonal patterns, demographic changes, or updated equipment. This is **covariate shift**: features change while the label relationship stays the same.

We simulate three monthly batches by adding Gaussian noise with increasing mean shift:

| Batch | Mean shift | Represents |
|-------|-----------|------------|
| Month 1 | 0.0 | No drift |
| Month 2 | 0.1 | Low drift |
| Month 3 | 0.3 | High drift |

In [3]:
# Simulate 3 monthly batches with increasing covariate shift
np.random.seed(42)
batches      = []
drift_levels = [0.0, 0.1, 0.3]

for i, drift in enumerate(drift_levels):
    noise = np.random.normal(loc=drift, scale=0.1, size=X_test.shape)
    batch = X_test + noise
    batches.append(batch)
    print(f"Batch {i+1} (Month {i+1}) | drift={drift} | mean shift: {noise.mean():.4f}")

Batch 1 (Month 1) | drift=0.0 | mean shift: 0.0026
Batch 2 (Month 2) | drift=0.1 | mean shift: 0.0960
Batch 3 (Month 3) | drift=0.3 | mean shift: 0.3005


---
## Section 3 — Evidently Data Drift Report

Evidently runs statistical tests per feature to detect drift from the reference (training) data:
- **Continuous features:** Kolmogorov-Smirnov (KS) test
- **Categorical features:** Chi-squared test

We compare training data against Month 3 (highest drift) and save a full HTML report.

In [ ]:
from evidently import Report
from evidently.presets import DataDriftPreset

# Compare training data (reference) vs most drifted batch (Month 3)
report = Report([DataDriftPreset()])
result = report.run(
    reference_data=X_train,
    current_data=batches[2]
)

result.save_html("reports/data_drift_report.html")
print("Drift report saved to: reports/data_drift_report.html")
print("Open this file in a browser to see feature-level drift analysis.")

---
## Section 4 — Track Performance Across Batches

We evaluate the original model on each incoming batch and log results to MLflow as separate monitoring runs. The same model and same true labels are used across all batches — only the input features change, so drift is the sole cause of any performance change.

In [ ]:
# Evaluate model on each batch and log to MLflow
performance_log = []

for i, batch in enumerate(batches):
    batch_preds = rf.predict(batch)
    batch_acc   = accuracy_score(y_test, batch_preds)
    batch_auc   = roc_auc_score(y_test, rf.predict_proba(batch)[:, 1])

    performance_log.append({
        'batch':       f'Month {i+1}',
        'accuracy':    round(batch_acc, 4),
        'auc_roc':     round(batch_auc, 4),
        'drift_level': ['none', 'low', 'high'][i]
    })

    with mlflow.start_run(run_name=f"batch_{i+1}_monitoring"):
        mlflow.log_metric("accuracy",    batch_acc)
        mlflow.log_metric("auc_roc",     batch_auc)
        mlflow.log_param("drift_level",  ['none', 'low', 'high'][i])

perf_df = pd.DataFrame(performance_log)
print("Performance across batches:\n")
print(perf_df.to_string(index=False))
print("\nAll monitoring runs logged to MLflow.")

---
## Section 5 — Visualise Performance Degradation

The red dashed line marks `RETRAIN_THRESHOLD = 0.90`. Note that **accuracy collapses** under high drift while **AUC-ROC stays high** — because AUC-ROC measures ranking ability rather than calibration. This is why AUC-ROC alone is insufficient for production monitoring.

In [ ]:
RETRAIN_THRESHOLD = 0.90

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Model Performance Over Time Under Increasing Data Drift", fontsize=13)

for ax, col, colour, label in [
    (axes[0], 'accuracy', '#7c6ef0', 'Accuracy'),
    (axes[1], 'auc_roc',  '#f472b6', 'AUC-ROC')
]:
    ax.plot(perf_df['batch'], perf_df[col],
            marker='o', color=colour, linewidth=2.5, markersize=9, label=label)
    ax.axhline(y=RETRAIN_THRESHOLD, color='red', linestyle='--',
               alpha=0.7, linewidth=1.5, label=f'Threshold ({RETRAIN_THRESHOLD})')
    ax.set_title(f'{label} Over Time')
    ax.set_ylabel('Score')
    ax.set_ylim(0.5, 1.02)
    ax.legend()
    ax.grid(True, alpha=0.3)
    for _, row in perf_df.iterrows():
        ax.annotate(f"{row[col]:.3f}",
                    xy=(row['batch'], row[col]),
                    xytext=(0, 10), textcoords='offset points',
                    ha='center', fontsize=9, color=colour)

plt.tight_layout()
plt.savefig('plots/performance_monitoring.png', dpi=150, bbox_inches='tight')
plt.show()
print("Plot saved to: plots/performance_monitoring.png")

---
## Section 6 — Retrain Trigger Logic

Checks each batch's AUC-ROC against the threshold and fires an alert if performance has dropped. In a real production system this would call a retraining job, open a ticket, or send a Slack alert.

In [ ]:
RETRAIN_THRESHOLD = 0.90

print(f"Retrain threshold: AUC-ROC < {RETRAIN_THRESHOLD}\n")
print("-" * 65)

for _, row in perf_df.iterrows():
    if row['auc_roc'] < RETRAIN_THRESHOLD:
        status = "--> RETRAIN RECOMMENDED"
        flag   = "ALERT"
    else:
        status = "Model within acceptable range"
        flag   = "OK   "

    print(f"  {flag}  {row['batch']} | Drift: {row['drift_level']:<4} | "
          f"Accuracy: {row['accuracy']:.4f} | AUC: {row['auc_roc']:.4f} | {status}")

print("-" * 65)
print("\nNote: AUC-ROC stays high under covariate shift because the model")
print("preserves ranking ability even as absolute calibration degrades.")
print("Monitor both metrics in production, not just AUC-ROC.")

---
## Summary

| Section | What it does |
|---|---|
| 1 | Trains Random Forest baseline, logs to MLflow |
| 2 | Simulates 3 months of incoming data with increasing drift |
| 3 | Generates Evidently HTML drift report (feature-level analysis) |
| 4 | Evaluates model per batch, logs monitoring runs to MLflow |
| 5 | Visualises accuracy and AUC-ROC degradation with threshold lines |
| 6 | Fires retrain alerts based on configurable AUC-ROC threshold |

**Key results**

| Batch | Accuracy | AUC-ROC | Drift |
|---|---|---|---|
| Month 1 | 0.9561 | 0.9805 | None |
| Month 2 | 0.9035 | 0.9858 | Low |
| Month 3 | 0.6053 | 0.9912 | High |

**Key insight:** Accuracy collapsed from 96% to 60% under high drift while AUC-ROC stayed above 0.99. This demonstrates why AUC-ROC alone is insufficient for production monitoring.

---